In [196]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
con = duckdb.connect("warehouse.duckdb")

## 3. Crear el esquema estrella en DuckDB

In [197]:
con.execute("""
    DROP TABLE IF EXISTS fact_sales;
    DROP TABLE IF EXISTS dim_time;
    DROP TABLE IF EXISTS dim_item;
    DROP TABLE IF EXISTS dim_location;
            
    -- DIMENSIONES — atributos descriptivos y jerarquías
    CREATE TABLE dim_time (
    time_key INTEGER PRIMARY KEY,
    day DATE, month VARCHAR, quarter VARCHAR, year INTEGER );
            
    CREATE TABLE dim_item (
    item_key INTEGER PRIMARY KEY,
    item_name VARCHAR, brand VARCHAR, category VARCHAR );
            
    CREATE TABLE dim_location (
    loc_key INTEGER PRIMARY KEY,
    city VARCHAR, state VARCHAR, country VARCHAR );
    -- HECHOS — claves foraneas + medidas numericas
            
    CREATE TABLE fact_sales (
    time_key INTEGER REFERENCES dim_time(time_key),
    item_key INTEGER REFERENCES dim_item(item_key),
    loc_key INTEGER REFERENCES dim_location(loc_key),
    dollars_sold DECIMAL(12,2),
    units_sold INTEGER );
""")

In [198]:
con.execute("""
    INSERT INTO dim_time SELECT * FROM read_csv_auto('data/dim_time.csv');
    INSERT INTO dim_item SELECT * FROM read_csv_auto('data/dim_item.csv');
    INSERT INTO dim_location SELECT * FROM read_csv_auto('data/dim_location.csv');
    INSERT INTO fact_sales SELECT * FROM read_csv_auto('data/fact_sales.csv');
""")

In [199]:
con.execute("""
    DROP VIEW IF EXISTS v_sales;
    CREATE VIEW v_sales AS
    SELECT t.year, t.quarter, t.month, i.category, i.item_name, l.country, f.dollars_sold, f.units_sold
    FROM fact_sales f
    JOIN dim_time t USING (time_key)
    JOIN dim_item i USING (item_key)
    JOIN dim_location l USING (loc_key);
""")

## 4. Operaciones OLAP — código ejecutable

## Roll-up y drill-down

Roll-up y drill-down son para escoger el nivel de los datos de una cara o dimensión. En este caso indica que tanto detalle va a tener la informacion que se va a mostrar, Roll up es para menos detalles tomando un nivel mas general, mientras que drill-down es tomar mas detalle.

Ejemplos:
Roll-up: pasar de cities y country en la consulta a solo country.
Drill-down: pasar year a year, month y quarter.

In [200]:
RollUp = con.execute("""
    SELECT country, SUM(dollars_sold), SUM(units_sold) FROM v_sales GROUP BY country;

""").df()
RollUp

,country,sum(dollars_sold),sum(units_sold)
0,USA,371309.46,11613.0
1,Mexico,382387.89,14230.0
2,Colombia,184710.89,7807.0
3,Chile,149523.18,5246.0


**Resultado:** 
La consulta nos va a otorgar la cantidad de dolares y unidades vendidas agrupados por el pais.

In [201]:
drillDown = con.execute("""
    SELECT year, month, quarter, SUM(dollars_sold), SUM(units_sold) FROM v_sales
    GROUP BY year, month, quarter;
""").df()
drillDown

,year,month,quarter,sum(dollars_sold),sum(units_sold)
0,2025,6,2,89586.91,3338.0
1,2025,3,1,91386.51,3346.0
2,2025,5,2,85397.28,3121.0
3,2025,10,4,91927.11,3319.0
4,2025,9,3,84327.79,3113.0
5,2025,4,2,91053.99,3330.0
6,2025,8,3,88701.09,3309.0
7,2025,12,4,111041.98,3323.0
8,2025,11,4,97816.87,3018.0
9,2025,1,1,83436.45,3163.0


**Resultado:** 
La consulta nos va a otorgar la cantidad de dolares y unidades vendidas agrupadas por año, mes y quarter

## Slice & dice
Sirve para obtener los datos de una o varias caras/dimensiones del cubo. 

In [202]:
Slice_Dice =con.execute("""
    SELECT category, SUM(dollars_sold), SUM(units_sold) FROM v_sales
    WHERE country = 'USA' GROUP BY category;
""").df()
Slice_Dice

,category,sum(dollars_sold),sum(units_sold)
0,Electronics,47506.61,1416.0
1,Home,64135.93,1927.0
2,Grocery,114985.81,3601.0
3,Beauty,78809.76,2665.0
4,Sports,65871.35,2004.0


**Resultado:** La consulta nos otorga cuanto se vendio con respecto a las variables (dollars_sold y units_sold) por categorias en el pais de USA

## Pivot
Cambia la forma en como se ve el resultado, intercambia las filas por las columnas.

In [203]:
pivot = con.execute("""
    PIVOT v_sales ON country USING SUM(dollars_sold) GROUP BY category;
""").df()
pivot

,category,Chile,Colombia,Mexico,USA
0,Electronics,19559.61,21901.29,51265.67,47506.61
1,Home,25176.05,31758.18,61481.04,64135.93
2,Beauty,32357.55,40182.81,75143.06,78809.76
3,Sports,26312.22,30887.24,71760.33,65871.35
4,Grocery,46117.75,59981.37,122737.79,114985.81


**Resultado:** Lo que hace es mostrarnos las ventas que hay por paises agrupados por categorias, entonces en las columnas nos mostrara a los paises y en las filas las categorias

## 5. Materialización con CUBE y ROLLUP

## CUBE

In [204]:
cube_df = con.execute("""
    SELECT year, country, category,
        SUM(dollars_sold) AS dollars_sold,
        SUM(units_sold) AS units_sold
    FROM v_sales
    GROUP BY CUBE (year, country, category)
""").df()

levels = con.execute("""
    SELECT
        COUNT(DISTINCT year) AS l_year,
        COUNT(DISTINCT country) AS l_country,
        COUNT(DISTINCT category) AS l_category
    FROM v_sales
""").df()

cube_df

,year,country,category,dollars_sold,units_sold
0,<NA>,NaN,NaN,1087931.42,38896.0
1,2025,NaN,NaN,1087931.42,38896.0
2,2025,USA,NaN,371309.46,11613.0
3,2025,Chile,Grocery,46117.75,1632.0
4,2025,Chile,Home,25176.05,822.0
5,2025,Chile,NaN,149523.18,5246.0
6,2025,Mexico,NaN,382387.89,14230.0
7,2025,Colombia,NaN,184710.89,7807.0
8,2025,Mexico,Electronics,51265.67,1865.0
9,2025,USA,Beauty,78809.76,2665.0


## ROLLUP

In [205]:
rollup_df = con.execute("""
    SELECT year, country, category, SUM(dollars_sold) AS dollars_sold, SUM(units_sold) AS units_sold
    FROM v_sales
    GROUP BY ROLLUP (year, country, category)
""").df()
rollup_df

,year,country,category,dollars_sold,units_sold
0,<NA>,NaN,NaN,1087931.42,38896.0
1,2025,NaN,NaN,1087931.42,38896.0
2,2025,Colombia,NaN,184710.89,7807.0
3,2025,Mexico,NaN,382387.89,14230.0
4,2025,Chile,NaN,149523.18,5246.0
5,2025,USA,NaN,371309.46,11613.0
6,2025,Chile,Grocery,46117.75,1632.0
7,2025,Chile,Home,25176.05,822.0
8,2025,Mexico,Electronics,51265.67,1865.0
9,2025,USA,Beauty,78809.76,2665.0


## 6. Distributiva vs holística — la comparación obligatoria

## GROUPING SETS con SUM

In [206]:
groupset_sums_df = con.execute("""
    SELECT year, country, SUM(dollars_sold), SUM(units_sold)
    FROM v_sales
    GROUP BY GROUPING SETS ((year), (country), (year,country), ());
""").df()
groupset_sums_df

,year,country,sum(dollars_sold),sum(units_sold)
0,<NA>,USA,371309.46,11613.0
1,2025,NaN,1087931.42,38896.0
2,<NA>,Chile,149523.18,5246.0
3,2025,Mexico,382387.89,14230.0
4,<NA>,Mexico,382387.89,14230.0
5,2025,Chile,149523.18,5246.0
6,2025,USA,371309.46,11613.0
7,2025,Colombia,184710.89,7807.0
8,<NA>,NaN,1087931.42,38896.0
9,<NA>,Colombia,184710.89,7807.0


## GROUPING SETS con MEDIAN


In [207]:
groupset_median_df = con.execute("""
    SELECT year, country,   MEDIAN(dollars_sold), MEDIAN(units_sold)
    FROM v_sales
    GROUP BY GROUPING SETS ((year), (country), (year,country), ());
""").df()
groupset_median_df

,year,country,median(dollars_sold),median(units_sold)
0,<NA>,Chile,93.42,3.0
1,2025,USA,104.48,3.0
2,2025,NaN,91.31,3.0
3,<NA>,Mexico,90.17,4.0
4,2025,Chile,93.42,3.0
5,<NA>,Colombia,75.83,3.0
6,2025,Mexico,90.17,4.0
7,2025,Colombia,75.83,3.0
8,<NA>,USA,104.48,3.0
9,<NA>,NaN,91.31,3.0


**Respuesta**
• ¿Cuál es más rápida? Mide con %timeit o time.perf_counter().
• ¿Cuál se puede particionar? ¿Por qué SUM se puede agregar incrementalmente y MEDIAN no?
• Si tuvieras 1,000 millones de filas, ¿cuál materializarías y cuál calcularías al vuelo? ¿Por qué?

## Iceberg cube

In [209]:
iceberg_df = con.execute("""
        select country, month, item_name, SUM(dollars_sold) AS dollars_sold
        FROM v_sales 
        Group by cube (country, month, item_name)
        having SUM(dollars_sold) >= 50000
""").df()
iceberg_df


,country,month,item_name,dollars_sold
0,USA,NaN,None,371309.46
1,NaN,8,None,88701.09
2,NaN,12,None,111041.98
3,NaN,2,None,81791.69
4,NaN,7,None,91463.75
5,NaN,5,None,85397.28
6,NaN,9,None,84327.79
7,NaN,6,None,89586.91
8,NaN,10,None,91927.11
9,Colombia,NaN,None,184710.89
